# Chapter 38 — Representing Images and Geometry

Companion tutorial for **[Foundations of Computer Vision](https://visionbook.mit.edu/homogeneous_coordinates.html)** by Torralba, Isola, and Freeman (MIT Press, 2024).

This is the foundational chapter for everything in Part XI (geometry). Three ideas drive it:

1. **Homogeneous coordinates** let you write translation, rotation, scaling, shearing, and projective warp as a single matrix multiplication.
2. **Lines and points are dual** under the cross product in homogeneous form — joining two points and intersecting two lines are the same operation.
3. **Implicit image representations** (SIREN-style MLPs that map $(x,y)\to \text{RGB}$) give you sub-pixel image access without any interpolation kernel — geometric transformations become inverse-coordinate evaluations.

We reproduce all 12 figures. The most interesting bits live at the ends: figure 38.8 reframes warping as a 1D convolution on coordinates, and figure 38.12 swaps the entire pixel grid for a neural network.

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn as nn
import kornia.geometry.transform as KT
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle, Circle, FancyArrowPatch, PathPatch
from matplotlib.path import Path as MplPath
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from PIL import Image, ImageDraw
import skimage.data

torch.set_default_dtype(torch.float32)
torch.manual_seed(0)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": False,
})

## 38.2 — Homogeneous and heterogeneous coordinates

Heterogeneous coordinates write a 2D point as $(x, y)$. Homogeneous coordinates write the same point as $(x, y, 1)$ — and crucially, also as $(\lambda x, \lambda y, \lambda)$ for any $\lambda \ne 0$. All points on the ray through the origin and $(x, y, 1)$ represent the same 2D point. Conversion back is divide-by-$w$:

$$\begin{bmatrix}x\\y\\w\end{bmatrix} \;\longrightarrow\; \left(\frac{x}{w}, \frac{y}{w}\right).$$

The payoff: translation, which is an *addition* in heterogeneous coords, becomes a *multiplication* in homogeneous coords. That uniformity lets you compose any sequence of geometric transformations as a single matrix product.

In [ ]:
def to_homogeneous(p):
    """Append a 1 to the last axis.  (..., D) -> (..., D+1)."""
    return torch.cat([p, torch.ones_like(p[..., :1])], dim=-1)


def from_homogeneous(p):
    """Divide by last coord. (..., D+1) -> (..., D)."""
    return p[..., :-1] / p[..., -1:]

In [ ]:
# Figure 38.1 - heterogeneous plane (left) and the w=1 plane (middle).
# A point (x, y) on the heterogeneous plane corresponds to (x, y, 1) on the
# w=1 plane, and to (lambda*x, lambda*y, lambda) for any lambda != 0 on the
# dashed ray extending outward.
fig = plt.figure(figsize=(9, 5.2))
ax = fig.add_subplot(111, projection="3d")

# Heterogeneous plane: x-y plane at w=0 (vertical, on the left)
het_plane = [
    [0, -0.4, -0.4], [0, 1.8, -0.4], [0, 1.8, 1.8], [0, -0.4, 1.8],
]
ax.add_collection3d(Poly3DCollection([het_plane], facecolor="#b5d3c4", alpha=0.45, edgecolor="#345f4d"))

# w=1 plane: vertical, at w=1
w1_plane = [
    [1, -0.4, -0.4], [1, 1.8, -0.4], [1, 1.8, 1.8], [1, -0.4, 1.8],
]
ax.add_collection3d(Poly3DCollection([w1_plane], facecolor="#dddddd", alpha=0.35, edgecolor="#666666"))

# Axes from origin
ax.plot([0, 2.6], [0, 0], [0, 0], "k", lw=1.2)     # w-axis
ax.plot([0, 0], [0, 1.7], [0, 0], "k", lw=1.2)     # y-axis (in-page)
ax.plot([0, 0], [0, 0], [0, 1.7], "k", lw=1.2)     # x-axis (vertical at left)
ax.text(2.7, 0, 0, "w", fontsize=13)
ax.text(0, 1.85, 0, "y", fontsize=13)
ax.text(0, 0, 1.85, "x", fontsize=13)
ax.text(1.0, 0.05, -0.20, "1", fontsize=11)

# Point (x, y) on the heterogeneous plane (w=0)
x_val, y_val = 1.05, 0.55
ax.scatter([0], [y_val], [x_val], color="#d62728", s=70, zorder=6)
ax.text(0, y_val + 0.06, x_val - 0.15, "(x,y)", fontsize=11)

# Point (x, y, 1) on the w=1 plane
ax.scatter([1], [y_val], [x_val], color="#d62728", s=80, zorder=6)
ax.text(1.05, y_val + 0.04, x_val - 0.20, "(x,y,1)", fontsize=11)

# Point (lambda*x, lambda*y, lambda) far on the ray
lam = 2.2
ax.scatter([lam], [y_val * lam], [x_val * lam], color="#d62728", s=70, zorder=6)
ax.text(lam * 1.02, y_val * lam, x_val * lam + 0.08, r"$(\lambda x, \lambda y, \lambda)$", fontsize=11)

# Solid line from (x,y) to (x,y,1) -- equivalence between the two reps
ax.plot([0, 1], [y_val, y_val], [x_val, x_val], color="#444", lw=0.9)

# Dashed ray from origin through (x,y,1) out to (lambda*..., ..., ...)
lams = torch.linspace(0, lam * 1.05, 50)
ax.plot(lams.tolist(),
        (lams * y_val).tolist(),
        (lams * x_val).tolist(),
        "k--", lw=0.9)

# Space labels
ax.text(-0.05, -0.65, 0.9, "Heterogeneous", fontsize=12, color="#222")
ax.text(2.0, -0.65, 1.1, "Homogeneous", fontsize=12, color="#222")

ax.set_xlim(-0.1, 3.0); ax.set_ylim(-0.6, 2.0); ax.set_zlim(-0.4, 2.2)
ax.set_box_aspect((1.6, 1.4, 1.4))
ax.view_init(elev=18, azim=-75)
ax.set_axis_off()
ax.set_title("Figure 38.1 - heterogeneous and homogeneous coordinates")
plt.tight_layout()
plt.show()

## 38.3 — 2D image transformations

Every transformation that follows is a $3\times 3$ matrix acting on homogeneous points. The book applies each transformation to a photograph of a clock; we draw a clock face procedurally, rasterise it to a tensor, then apply the transformation as an actual image warp with `kornia.geometry.transform.warp_affine` — so what you see is the matrix acting on real pixels.

In [ ]:
# Reference object: a procedurally-drawn clock face, rasterised to a torch tensor.
# The book uses a photograph of a wall clock and applies each transformation to
# it; this stand-in serves the same pedagogical role without copyright concerns.
def apply_T(M, pts):
    """Apply 3x3 homogeneous matrix to (N, 2) points - still used by the
    hand-written forward_warp / backward_warp helpers in the warping section."""
    return from_homogeneous(to_homogeneous(pts) @ M.T)


ASSET_DIR = Path("assets")  # relative to the notebook directory at execution time


def render_clock(size=256):
    """Load the bundled wall-clock asset (assets/wallclock.png) as-is.
    The asset already has natural background padding around the clock, so we
    just resize to `size` if needed and return the tensor unchanged."""
    img = Image.open(ASSET_DIR / "wallclock.png").convert("L")
    if img.size != (size, size):
        img = img.resize((size, size), Image.LANCZOS)
    pixels = list(img.getdata())
    return torch.tensor(pixels, dtype=torch.float32).view(size, size) / 255.0


CLOCK = render_clock(256)
CLOCK_H, CLOCK_W = CLOCK.shape


def warp_image(img, M_3x3, dsize=None):
    """Apply a 3x3 homogeneous matrix to an image via kornia (backward, bilinear).
    Convention: M maps source pixel coords to target coords."""
    H, W = img.shape
    if dsize is None:
        dsize = (H, W)
    img_bchw = img.unsqueeze(0).unsqueeze(0)
    M_b23 = M_3x3[:2, :3].unsqueeze(0)
    out = KT.warp_affine(img_bchw, M_b23, dsize=dsize, mode="bilinear", padding_mode="border")
    return out.squeeze(0).squeeze(0)


def show_clock(ax, img, title):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)


def T_translate(tx, ty):
    return torch.tensor([[1.0, 0.0, tx], [0.0, 1.0, ty], [0.0, 0.0, 1.0]])


def T_scale(sx, sy, cx=0.0, cy=0.0):
    """Scale about the point (cx, cy)."""
    return (
        torch.tensor([[1.0, 0.0, cx], [0.0, 1.0, cy], [0.0, 0.0, 1.0]])
        @ torch.tensor([[sx, 0.0, 0.0], [0.0, sy, 0.0], [0.0, 0.0, 1.0]])
        @ torch.tensor([[1.0, 0.0, -cx], [0.0, 1.0, -cy], [0.0, 0.0, 1.0]])
    )


def T_rotate(theta, cx=0.0, cy=0.0):
    """Rotation about (cx, cy) by theta radians (image-down y)."""
    c, s = math.cos(theta), math.sin(theta)
    return (
        torch.tensor([[1.0, 0.0, cx], [0.0, 1.0, cy], [0.0, 0.0, 1.0]])
        @ torch.tensor([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])
        @ torch.tensor([[1.0, 0.0, -cx], [0.0, 1.0, -cy], [0.0, 0.0, 1.0]])
    )


def T_shear(shx, shy):
    return torch.tensor([[1.0, shx, 0.0], [shy, 1.0, 0.0], [0.0, 0.0, 1.0]])


def T_projective_warp(strength=0.0028):
    """A homography that visibly tilts the image (right side closer to camera).
    Bigger strength -> more pronounced trapezoidal warp."""
    M = torch.eye(3)
    M[2, 0] = strength            # right side shrinks toward vanishing point
    M[2, 1] = -strength * 0.55    # top tilts inward
    return M


# Inverse of each transform is what warp_affine wants (backward mapping),
# but kornia's warp_affine expects the source-to-target matrix and inverts
# internally. We pass the forward transform.
cx, cy = CLOCK_W / 2, CLOCK_H / 2

# --- 2D axis-frame helpers (for figures 38.3-38.7 which use orange/green squares + xy axes) ---
def draw_xy_axes(ax, lim=2.2, axis_color="black", lw=1.4, tick=1.0,
                 show_ticks=True, tick_labels=("-1", "1")):
    """Draw x and y axes with arrowheads through the origin; optional -1 / 1 ticks."""
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.axis("off")
    ax.add_patch(FancyArrowPatch((-lim, 0), (lim, 0), color=axis_color, lw=lw,
                                 arrowstyle="-|>", mutation_scale=14, zorder=6))
    ax.add_patch(FancyArrowPatch((0, -lim), (0, lim), color=axis_color, lw=lw,
                                 arrowstyle="-|>", mutation_scale=14, zorder=6))
    ax.text(lim - 0.05, -0.18, "$x$", fontsize=15, ha="right", va="top", zorder=7)
    ax.text(-0.10, lim - 0.05, "$y$", fontsize=15, ha="right", va="top", zorder=7)
    if show_ticks:
        for v, lab in zip([-tick, tick], tick_labels):
            ax.plot([v, v], [-0.07, 0.07], color=axis_color, lw=lw, zorder=6)
            ax.plot([-0.07, 0.07], [v, v], color=axis_color, lw=lw, zorder=6)
            ax.text(v, -0.18, lab, fontsize=12, ha="center", va="top", zorder=7)
            ax.text(-0.14, v, lab, fontsize=12, ha="right", va="center", zorder=7)


def unit_square(transform=None, scale_xy=(1.0, 1.0)):
    """Return (4, 2) corners of a unit square centred at origin, optionally
    after applying a 3x3 homogeneous transform (multiplied left, x' = M @ x)."""
    sx, sy = scale_xy
    corners = torch.tensor([
        [-sx, -sy], [sx, -sy], [sx, sy], [-sx, sy],
    ], dtype=torch.float32)
    if transform is not None:
        homog = torch.cat([corners, torch.ones(4, 1)], dim=-1)
        out = (transform @ homog.T).T
        return out[:, :2] / out[:, 2:3]
    return corners


def filled_square(ax, pts, facecolor="#f59e2c", edgecolor="black", lw=1.6, alpha=1.0, zorder=2):
    ax.add_patch(Polygon(pts.tolist(), facecolor=facecolor, edgecolor=edgecolor,
                         lw=lw, alpha=alpha, zorder=zorder))


In [ ]:
# Figure 38.2 - six clock panels: reference + translation + rotation + scaling + shear + warping.
# Each transformation is applied to the clock IMAGE via kornia.warp_affine /
# warp_perspective (so what you see is the matrix multiplying real pixels).
# The clock is rendered with 14% padding around it so shears and rotations don't
# clip the clock face in the output frame.
fig, axes = plt.subplots(1, 6, figsize=(13.5, 2.6))

show_clock(axes[0], CLOCK, "reference")

M_tr = T_translate(-32, 18)
show_clock(axes[1], warp_image(CLOCK, M_tr), "translation")

M_rot = T_rotate(math.pi / 7, cx=cx, cy=cy)
show_clock(axes[2], warp_image(CLOCK, M_rot), "rotation")

M_sc = T_scale(1.35, 1.35, cx=cx, cy=cy)
show_clock(axes[3], warp_image(CLOCK, M_sc), "scaling")

M_sh = T_translate(-0.30 * cy, 0) @ T_shear(0.30, 0.0)
show_clock(axes[4], warp_image(CLOCK, M_sh), "shear")

# "Warping" panel: a true non-linear deformation (not a homography).  We build
# a smooth deterministic sinusoidal displacement field in pixel units and apply
# it with kornia.geometry.transform.remap, so the clock face becomes visibly
# wavy (clearly distinct from shear/perspective).
H, W = CLOCK.shape
yy_pix, xx_pix = torch.meshgrid(torch.arange(H).float(), torch.arange(W).float(), indexing="ij")
amp = 16.0   # max displacement in pixels
dx_pix = amp * torch.sin(2 * math.pi * yy_pix / H * 1.4)
dy_pix = amp * torch.cos(2 * math.pi * xx_pix / W * 1.4)
map_x = (xx_pix + dx_pix).unsqueeze(0)
map_y = (yy_pix + dy_pix).unsqueeze(0)
img_bchw = CLOCK.unsqueeze(0).unsqueeze(0)
warped = KT.remap(img_bchw, map_x, map_y, mode="bilinear", padding_mode="border")
show_clock(axes[5], warped.squeeze(0).squeeze(0), "warping")

fig.suptitle("Figure 38.2 - 2D geometric transformations applied to an image", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 38.3 - single translation; composition of two translations.
# Orange unit square at the origin + red translation vectors, matching the book.
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))

# Translation vectors (in axis units, NOT pixel units)
t = torch.tensor([1.0, 0.8])
s = torch.tensor([1.0, -0.4])

# LEFT: single translation
ax = axes[0]
draw_xy_axes(ax, lim=2.6, show_ticks=False)
# original square centred at origin, half-width 0.4 (matches book proportions)
sq = unit_square(scale_xy=(0.4, 0.4))
filled_square(ax, sq, facecolor="#f59e2c")
# translated copy
sq_t = sq + t
filled_square(ax, sq_t, facecolor="#f59e2c")
ax.add_patch(FancyArrowPatch((0, 0), (t[0].item(), t[1].item()), color="#e11d48", lw=2.4,
                             arrowstyle="-|>", mutation_scale=18, zorder=5))
ax.text(t[0].item() * 0.45 - 0.05, t[1].item() * 0.45 + 0.1, "$\\mathbf{t}$",
        fontsize=18, fontweight="bold")
ax.set_title("Translation", fontsize=12)

# RIGHT: composition t then s
ax = axes[1]
draw_xy_axes(ax, lim=3.0, show_ticks=False)
sq0 = unit_square(scale_xy=(0.4, 0.4))
sq1 = sq0 + t
sq2 = sq1 + s
filled_square(ax, sq0, facecolor="#f59e2c")
filled_square(ax, sq1, facecolor="#f59e2c")
filled_square(ax, sq2, facecolor="#f59e2c")
# vectors
ax.add_patch(FancyArrowPatch((0, 0), (t[0].item(), t[1].item()), color="#e11d48", lw=2.4,
                             arrowstyle="-|>", mutation_scale=18, zorder=5))
ax.add_patch(FancyArrowPatch((t[0].item(), t[1].item()),
                             ((t + s)[0].item(), (t + s)[1].item()),
                             color="#e11d48", lw=2.4,
                             arrowstyle="-|>", mutation_scale=18, zorder=5))
ax.add_patch(FancyArrowPatch((0, 0), ((t + s)[0].item(), (t + s)[1].item()),
                             color="#e11d48", lw=2.4,
                             arrowstyle="-|>", mutation_scale=18, zorder=5))
ax.text(t[0].item() * 0.45 - 0.15, t[1].item() * 0.45 + 0.1, "$\\mathbf{t}$",
        fontsize=18, fontweight="bold")
ax.text(t[0].item() + s[0].item() * 0.4 + 0.05, t[1].item() + s[1].item() * 0.4 + 0.05,
        "$\\mathbf{s}$", fontsize=18, fontweight="bold")
ax.text((t + s)[0].item() * 0.45 - 0.05, (t + s)[1].item() * 0.45 - 0.32,
        "$\\mathbf{t}+\\mathbf{s}$", fontsize=16, fontweight="bold")
ax.set_title("Composition", fontsize=12)

fig.suptitle("Figure 38.3 - translation and composition", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 38.4 - non-uniform (anisotropic) scaling on a unit square,
# rendered with the book's orange-square + xy-axes style.
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))

# LEFT: unit square at origin, half-width 1 (so corners at +/-1)
ax = axes[0]
draw_xy_axes(ax, lim=2.0)
filled_square(ax, unit_square(scale_xy=(1.0, 1.0)), facecolor="#f59e2c")

# RIGHT: scaled square: x'=2x, y'=0.5y
ax = axes[1]
draw_xy_axes(ax, lim=2.2)
M = T_scale(2.0, 0.5)
sq = unit_square(transform=M, scale_xy=(1.0, 1.0))
filled_square(ax, sq, facecolor="#f59e2c")
ax.text(0.7, 1.7, "$x'=2x$\n$y'=0.5y$", fontsize=14)

fig.suptitle("Figure 38.4 - non-uniform (anisotropic) scaling", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 38.5 - rotation by theta CCW around the origin, with dashed unit-circle
# guides and a theta arc, matching the book.
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))

ax = axes[0]
draw_xy_axes(ax, lim=2.0)
filled_square(ax, unit_square(scale_xy=(1.0, 1.0)), facecolor="#f59e2c")

ax = axes[1]
draw_xy_axes(ax, lim=2.0)
# dashed circumscribed circle (radius = sqrt(2) for unit square corners)
circ_th = torch.linspace(0, 2 * math.pi, 60).tolist()
ax.plot([math.sqrt(2) * math.cos(t) for t in circ_th],
        [math.sqrt(2) * math.sin(t) for t in circ_th],
        "k--", lw=0.7, alpha=0.6)
# arrowheads along the rotation direction (small arcs at +x and -x sides)
arc_t = torch.linspace(0, math.pi / 12, 8).tolist()
for centre_angle in [0.0, math.pi]:
    pts = [(math.sqrt(2) * math.cos(centre_angle + a), math.sqrt(2) * math.sin(centre_angle + a)) for a in arc_t]
    ax.plot([p[0] for p in pts], [p[1] for p in pts], "k", lw=1.2)
    # arrowhead at the tip
    head = pts[-1]; tail = pts[-3]
    ax.add_patch(FancyArrowPatch(tail, head, color="black", lw=0.0,
                                 arrowstyle="-|>", mutation_scale=12, zorder=4))
# rotated square
theta_show = math.pi / 9
M = T_rotate(theta_show)
sq = unit_square(transform=M, scale_xy=(1.0, 1.0))
filled_square(ax, sq, facecolor="#f59e2c")
# Solid diagonal line through origin at angle theta (the rotated x-axis),
# extending past the square in both directions - matches the book.
line_len = 1.9
ax.plot([-line_len * math.cos(theta_show), line_len * math.cos(theta_show)],
        [-line_len * math.sin(theta_show), line_len * math.sin(theta_show)],
        color="black", lw=1.2, zorder=5)
# theta arc between original x-axis and the rotated x-axis
arc_th = torch.linspace(0, theta_show, 18).tolist()
ax.plot([0.55 * math.cos(a) for a in arc_th],
        [0.55 * math.sin(a) for a in arc_th],
        color="black", lw=1.0)
ax.text(0.72, 0.06, r"$\theta$", fontsize=16, color="black", fontweight="bold")

fig.suptitle("Figure 38.5 - rotation by $\\theta$ around the origin", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 38.6 - four panels: reference + three shears
# (horizontal x'=x+y, vertical y'=x+y, combined), matching the book.
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))

panels = [
    ("reference",  None,                      None),
    ("$x'=x+y$\n$y'=y$", T_shear(1.0, 0.0), None),
    ("$x'=x$\n$y'=x+y$", T_shear(0.0, 1.0), None),
    ("combined",   T_shear(0.7, 0.7),         None),
]
for ax, (title, M, _) in zip(axes, panels):
    draw_xy_axes(ax, lim=2.5)
    if M is None:
        sq = unit_square(scale_xy=(1.0, 1.0))
    else:
        sq = unit_square(transform=M, scale_xy=(1.0, 1.0))
    filled_square(ax, sq, facecolor="#f59e2c")
    if "\n" in title:
        ax.text(0.7, 1.9, title, fontsize=13, va="top")
    else:
        ax.set_title(title, fontsize=12)

fig.suptitle("Figure 38.6 - three shear examples", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 38.7 - summary panel: green reference square + four transformed shapes
# (Translation, Scaling, Rotation, Shear) with their 3x3 matrix forms underneath.
fig, axes = plt.subplots(2, 5, figsize=(15.0, 6.0),
                          gridspec_kw={"height_ratios": [3, 1]})

panels = [
    ("",            None),
    ("Translation", T_translate(0.6, 0.5)),
    ("Scaling",     T_scale(1.4, 0.6)),
    ("Rotation",    T_rotate(math.pi / 8)),
    ("Shear",       T_shear(0.5, 0.0)),
]

# Plain ASCII matrices with Unicode prime, theta. The newlines are real
# multi-line strings; matplotlib treats them as line breaks in text.
TEXT_TR = "[x']   [1  0  tx] [x]\n[y'] = [0  1  ty] [y]\n[ 1]   [0  0   1] [1]"
TEXT_SC = "[x']   [sx 0  0] [x]\n[y'] = [0 sy  0] [y]\n[ 1]   [0  0  1] [1]"
TEXT_RO = "[x']   [cosθ -sinθ 0] [x]\n[y'] = [sinθ  cosθ 0] [y]\n[ 1]   [ 0      0     1] [1]"
TEXT_SH = "[x']   [1  qx 0] [x]\n[y'] = [qy 1  0] [y]\n[ 1]   [0  0  1] [1]"
matrices = [None, TEXT_TR, TEXT_SC, TEXT_RO, TEXT_SH]

for i, ((title, M), formula) in enumerate(zip(panels, matrices)):
    ax_top = axes[0, i]
    draw_xy_axes(ax_top, lim=1.9, show_ticks=False)
    if M is None:
        sq = unit_square(scale_xy=(0.6, 0.6))
    else:
        sq = unit_square(transform=M, scale_xy=(0.6, 0.6))
    filled_square(ax_top, sq, facecolor="#22c55e")
    if title:
        ax_top.set_title(title, fontsize=13, pad=4)
    if title == "Translation":
        # red translation arrow from origin to the centre of the translated square
        ax_top.add_patch(FancyArrowPatch((0, 0), (0.6, 0.5), color="#e11d48",
                                         lw=2.2, arrowstyle="-|>",
                                         mutation_scale=16, zorder=5))
        ax_top.text(0.05, 0.42, r"$\mathbf{t}$", fontsize=14, fontweight="bold", color="#e11d48")
    if title == "Rotation":
        # red-dashed line along the rotated x-axis + theta arc + label
        theta_show = math.pi / 8
        line_len = 1.4
        ax_top.plot([0, line_len * math.cos(theta_show)],
                    [0, line_len * math.sin(theta_show)],
                    color="#b91c1c", lw=1.2, linestyle="--", zorder=5)
        arc_th2 = torch.linspace(0, theta_show, 12).tolist()
        ax_top.plot([0.5 * math.cos(a) for a in arc_th2],
                    [0.5 * math.sin(a) for a in arc_th2],
                    color="#b91c1c", lw=1.0)
        ax_top.text(0.65, 0.05, r"$\theta$", fontsize=14, color="#b91c1c", fontweight="bold")

    ax_bot = axes[1, i]
    ax_bot.axis("off")
    if formula is not None:
        ax_bot.text(0.5, 0.5, formula, ha="center", va="center",
                    family="monospace", fontsize=10)

fig.suptitle("Figure 38.7 - summary of 2D geometric transformations", fontsize=12)
plt.tight_layout()
plt.show()

### 38.3.7 — Geometric transformation as a convolution

If you represent an image as a set of triples $\{(\ell_i, x_i, y_i)\}$ (intensity + explicit position), then a rotation is a *1D convolution on the position channels*:

$$\begin{bmatrix}x'\\y'\end{bmatrix} = \underbrace{\begin{bmatrix}\cos\theta & -\sin\theta\\\sin\theta & \cos\theta\end{bmatrix}}_{\text{kernels } w_x, w_y} \begin{bmatrix}x\\y\end{bmatrix}.$$

Intensity passes through untouched; only the coordinate channels are mixed (figure 38.8). This framing is what makes it natural to learn warps with neural networks operating on coordinate channels — the rest of the chapter sets up that bridge.

In [ ]:
# Figure 38.8 - geometric transformation reframed as a 1D convolution on coordinate
# channels. Following the book, we show TWO W_x/W_y kernel pairs (top and a middle
# focus row, with an ellipsis between them) to drive home that the convolution is
# applied independently at every pixel position. The intensity channel l_i passes
# through to the output unchanged.
fig, ax = plt.subplots(figsize=(11, 7.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 11)
ax.set_aspect("equal"); ax.axis("off")

n_rows = 9
row_y = torch.linspace(10.0, 1.6, n_rows).tolist()

col_xin  = {"x": 1.0, "y": 2.0, "l": 3.0}
col_xout = {"x": 11.0, "y": 12.0, "l": 13.0}

# Two W_x/W_y kernel pairs: at the top (row 1) and at the middle focus row.
focus_top = 1
focus_mid = n_rows // 2
kernel_offset = 0.55   # vertical offset between W_x and W_y within a pair

# Per-row colour palette so each pixel has its own (purple, magenta, cyan, black ...)
# - matches the book's heterogeneously-coloured nodes.
ROW_COLOURS_X = ["#ffabd1", "#ff5fae", "#ff2391", "#5b1238", "#000000",
                 "#5b1238", "#ff5fae", "#ffabd1", "#bcbec0"]
ROW_COLOURS_Y = ["#c7faff", "#7fe6e6", "#5fdfd0", "#205050", "#0a0a0a",
                 "#205050", "#5fdfd0", "#7fe6e6", "#c7faff"]
ROW_COLOURS_L = ["#dadcde", "#a8acb0", "#7c8084", "#42454a", "#000000",
                 "#42454a", "#7c8084", "#a8acb0", "#1f4cff"]
COL_KERNEL = "#bcd6ff"

def col_dot(ax, x, y, color, ring=False, zorder=3):
    ax.add_patch(Circle((x, y), 0.30, facecolor=color, edgecolor="black", lw=0.9, zorder=zorder))
    if ring:
        ax.add_patch(Circle((x, y), 0.42, facecolor="none", edgecolor="black", lw=1.4, zorder=zorder + 1))

# Input + output columns
for i, y in enumerate(row_y):
    ring = (i == focus_mid)
    col_dot(ax, col_xin["x"], y, ROW_COLOURS_X[i], ring=ring)
    col_dot(ax, col_xin["y"], y, ROW_COLOURS_Y[i], ring=ring)
    col_dot(ax, col_xin["l"], y, ROW_COLOURS_L[i], ring=ring)
    col_dot(ax, col_xout["x"], y, ROW_COLOURS_X[i], ring=ring)
    col_dot(ax, col_xout["y"], y, ROW_COLOURS_Y[i], ring=ring)
    col_dot(ax, col_xout["l"], y, ROW_COLOURS_L[i], ring=ring)

# Column labels
for col, lab in [(col_xin["x"], "$x$"), (col_xin["y"], "$y$"), (col_xin["l"], r"$\ell$"),
                 (col_xout["x"], "$x'$"), (col_xout["y"], "$y'$"), (col_xout["l"], r"$\ell'$")]:
    ax.text(col, 0.65, lab, ha="center", fontsize=15)

# Ellipsis dots between top focus row and middle focus row
for x in [col_xin["x"], col_xin["y"], col_xin["l"], col_xout["x"], col_xout["y"], col_xout["l"]]:
    for dy in [-0.18, 0.0, 0.18]:
        ax.text(x, (row_y[focus_top] + row_y[focus_mid]) / 2 + dy, ".", ha="center", va="center",
                fontsize=12, color="#888")

# Kernel-box rendering helper
def draw_kernel_pair(ax, y_centre, label_idx):
    wx_box = (6.4, y_centre + kernel_offset - 0.45, 1.6, 0.9)
    wy_box = (6.4, y_centre - kernel_offset - 0.45, 1.6, 0.9)
    for box, lab in [(wx_box, r"$\mathbf{w}_x$"), (wy_box, r"$\mathbf{w}_y$")]:
        x, y, w, h = box
        ax.add_patch(Rectangle((x, y), w, h, facecolor=COL_KERNEL, edgecolor="black",
                               lw=1.2, zorder=2))
        ax.text(x + w/2, y + h/2, lab, ha="center", va="center", fontsize=14)
    return wx_box, wy_box

def arrow(ax, src, dst, lw=1.0, mut=10):
    ax.add_patch(FancyArrowPatch(src, dst, arrowstyle="-|>", mutation_scale=mut,
                                 color="black", lw=lw, zorder=5, shrinkA=2, shrinkB=2))

# Draw both kernel pairs
for i in [focus_top, focus_mid]:
    y_i = row_y[i]
    wx_box, wy_box = draw_kernel_pair(ax, y_i, i)
    # x_i and y_i feed both W_x and W_y
    arrow(ax, (col_xin["x"], y_i), (wx_box[0], wx_box[1] + wx_box[3]/2))
    arrow(ax, (col_xin["y"], y_i), (wx_box[0], wx_box[1] + wx_box[3]/2))
    arrow(ax, (col_xin["x"], y_i), (wy_box[0], wy_box[1] + wy_box[3]/2))
    arrow(ax, (col_xin["y"], y_i), (wy_box[0], wy_box[1] + wy_box[3]/2))
    # kernel -> output
    arrow(ax, (wx_box[0] + wx_box[2], wx_box[1] + wx_box[3]/2), (col_xout["x"], y_i))
    arrow(ax, (wy_box[0] + wy_box[2], wy_box[1] + wy_box[3]/2), (col_xout["y"], y_i))

# Focus-row labels (the i-th pixel)
y_focus = row_y[focus_mid]
ax.text(col_xin["x"] - 0.55, y_focus, "$x_i$", fontsize=12, ha="right", va="center")
ax.text(col_xin["l"] + 0.5, y_focus, r"$\ell_i$", fontsize=12, va="center")
ax.text(col_xout["x"] - 0.55, y_focus, "$x_i'$", fontsize=12, ha="right", va="center")
ax.text(col_xout["l"] + 0.5, y_focus, r"$\ell_i'$", fontsize=12, va="center")

fig.suptitle("Figure 38.8 - rotation reframed as a 1D convolution on (x, y) channels", fontsize=12)
plt.tight_layout()
plt.show()

## 38.4 — Lines and points: cross-product duality

In homogeneous coordinates, a 2D line $ax + by + c = 0$ is the vector $\mathbf{l} = (a, b, c)$, and incidence is $\mathbf{l}^\top \mathbf{p} = 0$. Two operations turn out to be the **same** cross product:

$$\mathbf{l} = \mathbf{p}_1 \times \mathbf{p}_2 \qquad \text{(line through two points)}$$

$$\mathbf{p} = \mathbf{l}_1 \times \mathbf{l}_2 \qquad \text{(intersection of two lines)}$$

Parallel lines intersect at a point with $w = 0$ — a *point at infinity*, the homogeneous form of a vanishing point (this is exactly the construction Ch. 47 uses).

In [ ]:
# Figure 38.9 — line through two points, intersection of two lines (same operation).
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))

# Left: line through two points
p1 = torch.tensor([0.5, 0.7])
p2 = torch.tensor([3.0, 2.3])
line = torch.cross(to_homogeneous(p1), to_homogeneous(p2), dim=-1)
xs = torch.linspace(0, 3.5, 50)
ys = -(line[0].item() * xs + line[2].item()) / line[1].item()
axes[0].plot(xs, ys, "crimson", lw=1.2)
axes[0].scatter(*p1, color="navy", s=70, zorder=5)
axes[0].scatter(*p2, color="navy", s=70, zorder=5)
axes[0].text(*p1, r" $\mathbf{p}_1$", va="top")
axes[0].text(*p2, r" $\mathbf{p}_2$", va="top")
axes[0].set_title(r"$\mathbf{l} = \mathbf{p}_1 \times \mathbf{p}_2$")

# Right: intersection of two lines
l1 = torch.tensor([1.0, -0.5, -0.6])
l2 = torch.tensor([-0.3, 1.0, -1.2])
p_int = from_homogeneous(torch.cross(l1, l2, dim=-1))
for line_vec, color in [(l1, "crimson"), (l2, "seagreen")]:
    a, b, c = line_vec
    if abs(b.item()) > 1e-6:
        ys2 = -(a.item() * xs + c.item()) / b.item()
        axes[1].plot(xs, ys2, color=color, lw=1.2)
axes[1].scatter(*p_int, color="navy", s=80, zorder=5)
axes[1].text(*p_int, r" $\mathbf{p}=\mathbf{l}_1\times\mathbf{l}_2$", va="top")
axes[1].set_title(r"$\mathbf{p} = \mathbf{l}_1 \times \mathbf{l}_2$")

for ax in axes:
    ax.set_xlim(-0.2, 3.7); ax.set_ylim(-0.2, 3.0); ax.set_aspect("equal")
    ax.grid(True, lw=0.3)
fig.suptitle("Figure 38.9 — points and lines are dual under the cross product")
plt.tight_layout()
plt.show()

## 38.5 — Image warping: forward vs. backward mapping

Applying a transformation $M$ to an image *should* give you a new image. The naive way (forward mapping) walks input pixels, transforms each location, and writes intensity to the nearest target pixel — that leaves **holes**, because the inverse-density of an expansion never covers the target grid cleanly. The correct way (backward mapping) walks *target* pixels and pulls back through $M^{-1}$, so every output pixel gets one well-defined intensity (figures 38.10–38.11).

We'll demonstrate on a small checkerboard so the artifacts are easy to see.

In [ ]:
def make_checkerboard(n=80, squares=8):
    """Small checkerboard image as a tensor in [0, 1]."""
    g = (torch.arange(n)[:, None] // (n // squares)) + (torch.arange(n)[None, :] // (n // squares))
    return (g % 2).float()


def forward_warp(img, M):
    """Forward mapping with nearest-neighbour write. Produces holes."""
    H, W = img.shape
    out = torch.zeros_like(img)
    ys, xs = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    pts = torch.stack([xs.flatten().float(), ys.flatten().float()], dim=-1)
    new = apply_T(M, pts).round().long()
    new_x, new_y = new[:, 0], new[:, 1]
    mask = (new_x >= 0) & (new_x < W) & (new_y >= 0) & (new_y < H)
    out[new_y[mask], new_x[mask]] = img[ys.flatten()[mask], xs.flatten()[mask]]
    return out


def backward_warp(img, M):
    """Backward mapping with nearest-neighbour sample. Hole-free."""
    H, W = img.shape
    Minv = torch.linalg.inv(M)
    ys, xs = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
    pts = torch.stack([xs.flatten().float(), ys.flatten().float()], dim=-1)
    src = apply_T(Minv, pts).round().long()
    src_x, src_y = src[:, 0], src[:, 1]
    mask = (src_x >= 0) & (src_x < W) & (src_y >= 0) & (src_y < H)
    out = torch.zeros_like(img)
    out.view(-1)[mask] = img[src_y[mask], src_x[mask]]
    return out

In [ ]:
# Figures 38.10 / 38.11
# 38.10: a hand-drawn-style sketch of forward (missing pixel = "?") vs backward
#        (every target covered) mapping over a pair of pixel grids.
# 38.11: the SAME forward/backward warps applied to the CLOCK image (matches the
#        book's photographic example).
import itertools

def draw_grid(ax, ox, oy, n=8, w=0.45, color="black", lw=0.9):
    for i in range(n + 1):
        ax.plot([ox, ox + n * w], [oy + i * w, oy + i * w], color=color, lw=lw)
        ax.plot([ox + i * w, ox + i * w], [oy, oy + n * w], color=color, lw=lw)

def grid_centre(ox, oy, i, j, w=0.45):
    return ox + (j + 0.5) * w, oy + (i + 0.5) * w


# ---- 38.10 conceptual sketch ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
pairs_fw = [(1, 1, 1, 4), (3, 2, 2, 5), (5, 3, 4, 6), (2, 5, 3, 2), (4, 6, 5, 3)]
hole_target = (4, 3)
colors = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd", "#e377c2"]

ax = axes[0]
ax.set_aspect("equal"); ax.axis("off")
ax.set_xlim(-0.4, 8.5); ax.set_ylim(-0.6, 4.4)
draw_grid(ax, 0, 0); draw_grid(ax, 5, 0)
ax.text(0.45 * 8 / 2, -0.4, "source", ha="center", fontsize=10)
ax.text(5 + 0.45 * 8 / 2, -0.4, "target", ha="center", fontsize=10)
ax.set_title("Forward mapping", fontsize=11)
for (si, sj, ti, tj), col in zip(pairs_fw, colors):
    ax.add_patch(FancyArrowPatch(grid_centre(0, 0, si, sj), grid_centre(5, 0, ti, tj),
                                 arrowstyle="-|>", mutation_scale=12, color=col, lw=1.4,
                                 connectionstyle="arc3,rad=-0.25", zorder=5))
hx, hy = grid_centre(5, 0, *hole_target)
ax.add_patch(Rectangle((hx - 0.18, hy - 0.18), 0.36, 0.36,
                       facecolor="#ffe4e1", edgecolor="#aa0000", lw=1.2, zorder=4))
ax.text(hx, hy, "?", ha="center", va="center", fontsize=14, color="#aa0000", fontweight="bold")

# Backward panel - sparser arrows (one per row, plus a few extra) to read clearly
ax = axes[1]
ax.set_aspect("equal"); ax.axis("off")
ax.set_xlim(-0.4, 8.5); ax.set_ylim(-0.6, 4.4)
draw_grid(ax, 0, 0); draw_grid(ax, 5, 0)
ax.text(0.45 * 8 / 2, -0.4, "source", ha="center", fontsize=10)
ax.text(5 + 0.45 * 8 / 2, -0.4, "target", ha="center", fontsize=10)
ax.set_title("Backward mapping", fontsize=11)
pairs_bw = [(2, 5, 1, 2), (3, 3, 2, 4), (4, 6, 3, 1), (4, 2, 4, 5), (5, 4, 5, 3),
            (1, 4, 6, 2), (6, 5, 0, 1), (3, 1, 7, 4)]
for (si, sj, ti, tj), col in zip(pairs_bw, colors * 2):
    ax.add_patch(FancyArrowPatch(grid_centre(5, 0, ti, tj), grid_centre(0, 0, si, sj),
                                 arrowstyle="-|>", mutation_scale=12, color=col, lw=1.4,
                                 connectionstyle="arc3,rad=0.22", zorder=5))

fig.suptitle("Figure 38.10 - the conceptual difference between forward and backward mapping", fontsize=11)
plt.tight_layout()
plt.show()


# ---- 38.11 real warp applied to the CLOCK (matches the book's photographic example) ----
clock_img = CLOCK
M = T_rotate(math.pi / 8, cx=cx, cy=cy) @ T_scale(1.05, 1.05, cx=cx, cy=cy)

clock_fwd = forward_warp(clock_img, M)
clock_bwd = backward_warp(clock_img, M)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, im, title in zip(axes,
                         [clock_img, clock_fwd, clock_bwd],
                         ["original", "Forward mapping", "Backward mapping"]):
    ax.imshow(im, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Figure 38.11 - forward vs backward mapping on a real image", fontsize=11)
plt.tight_layout()
plt.show()

### Production-grade warp: `kornia.geometry.transform.warp_affine`

The hand-written `backward_warp` above is the principle. In real code you'd
reach for [`kornia`](https://kornia.readthedocs.io/) — the differentiable
computer-vision library on top of PyTorch. Its `warp_affine` does exactly the
backward mapping we just implemented, but with bilinear interpolation (so no
nearest-neighbour aliasing), batched, and differentiable end-to-end. We feed it
the same matrix $M$ and confirm the result lines up with our handwritten
version.

In [ ]:
# kornia equivalent of backward_warp - bilinear, batched, differentiable.
# We feed the clock image used above so the side-by-side keeps the same subject.
img_bchw = clock_img.unsqueeze(0).unsqueeze(0)
M_b23 = M[:2, :3].unsqueeze(0)
H, W = clock_img.shape
kornia_warp = KT.warp_affine(img_bchw, M_b23, dsize=(H, W), mode="bilinear")
kornia_warp = kornia_warp.squeeze(0).squeeze(0)

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.6))
axes[0].imshow(clock_bwd, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
axes[0].set_title("our backward_warp (nearest)")
axes[1].imshow(kornia_warp, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
axes[1].set_title("kornia.warp_affine (bilinear)")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Backward mapping: hand-rolled vs. kornia.geometry.transform.warp_affine")
plt.tight_layout()
plt.show()

## 38.6 — Implicit image representations (SIREN)

Backward mapping at sub-pixel locations needs an interpolation kernel — but only because the image is stored as a discrete grid. **SIREN** [Sitzmann et al., NeurIPS 2020] does away with the grid: it represents an image as a small MLP $f_\theta : (x, y) \to \text{intensity}$, where every layer uses $\sin$ activations. The trained network *is* the image — you can evaluate it at any continuous coordinate.

Geometric warping then becomes a coordinate transform with zero interpolation kernels:
$$\hat\ell(x, y) = f_\theta(M^{-1}(x, y, 1)^\top).$$

We train a tiny SIREN on a small synthetic test image and verify warping by feeding pre-transformed coordinates (figure 38.12). The training is intentionally short — the point is to demonstrate the representation, not push PSNR.

In [ ]:
class SIREN(nn.Module):
    """Tiny implicit image network. Sine-activated MLP per Sitzmann et al., 2020."""

    def __init__(self, hidden=256, layers=4, w0=30.0):
        super().__init__()
        self.w0 = w0
        net = [nn.Linear(2, hidden)]
        for _ in range(layers - 1):
            net.append(nn.Linear(hidden, hidden))
        net.append(nn.Linear(hidden, 1))
        self.net = nn.ModuleList(net)
        self._init_weights()

    def _init_weights(self):
        with torch.no_grad():
            self.net[0].weight.uniform_(-1 / 2, 1 / 2)
            for layer in self.net[1:]:
                bound = (6.0 / layer.in_features) ** 0.5 / self.w0
                layer.weight.uniform_(-bound, bound)

    def forward(self, xy):
        h = torch.sin(self.w0 * self.net[0](xy))
        for layer in self.net[1:-1]:
            h = torch.sin(self.w0 * layer(h))
        return self.net[-1](h)


def make_target_image(n=128):
    """Load the classic 'cameraman' test image (essentially what the book uses
    for Figure 38.12), centre-crop, and downsample to n x n in [0, 1]."""
    cam = skimage.data.camera()                  # (512, 512) uint8
    H, W = cam.shape
    # Centre-crop to square (already square, but stay explicit), then downsample
    side = min(H, W)
    cy0, cx0 = (H - side) // 2, (W - side) // 2
    cropped = cam[cy0:cy0 + side, cx0:cx0 + side]
    # Convert through PIL for nice resampling, then bytes -> torch (no numpy on the math side).
    img = Image.fromarray(cropped, mode="L").resize((n, n), Image.LANCZOS)
    pixels = list(img.getdata())
    return torch.tensor(pixels, dtype=torch.float32).view(n, n) / 255.0


def coord_grid(n):
    """Regular [-1, 1]^2 grid of shape (n*n, 2)."""
    ys = torch.linspace(-1, 1, n)
    xs = torch.linspace(-1, 1, n)
    yy, xx = torch.meshgrid(ys, xs, indexing="ij")
    return torch.stack([xx.flatten(), yy.flatten()], dim=-1)

In [ ]:
# Train SIREN on the cameraman image. Real images need more capacity / steps
# than a synthetic blob; 2000 steps with a slightly wider MLP gets a decent
# memorisation in ~10s on CPU.
N = 128
target = make_target_image(N).flatten().unsqueeze(-1)
grid = coord_grid(N)

model = SIREN(hidden=256, layers=4, w0=30.0)
optim = torch.optim.Adam(model.parameters(), lr=1e-4)

for step in range(2000):
    pred = model(grid)
    loss = ((pred - target) ** 2).mean()
    optim.zero_grad()
    loss.backward()
    optim.step()

print(f"final reconstruction MSE: {loss.item():.4f}")

In [ ]:
# Figure 38.12 - (a) SIREN-encoded image, (b) same network evaluated at rotated+scaled coords.
with torch.no_grad():
    encoded = model(grid).view(N, N).clamp(0, 1)

    theta = math.pi / 4
    M = T_rotate(theta) @ T_scale(0.5, 0.5)
    Minv = torch.linalg.inv(M)
    warped_coords = (Minv @ to_homogeneous(grid).T).T
    warped_coords = warped_coords[:, :2] / warped_coords[:, 2:3]
    warped = model(warped_coords).view(N, N).clamp(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.6))
axes[0].imshow(encoded, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("(a) image encoded by SIREN")
axes[1].imshow(warped, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("(b) rotated 45 deg + scaled 0.5x")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Figure 38.12 - implicit image, warped via inverse-coordinate evaluation", fontsize=11)
plt.tight_layout()
plt.show()

## 38.7 — Concluding remarks

The arc of the chapter is one promotion at a time:

* **Heterogeneous → homogeneous**: every affine and projective transformation becomes a single matrix product.
* **Points and lines → cross-product duality**: incidence, joins, and intersections collapse into one operation. Parallels meet at points with $w=0$ — vanishing points get a coordinate.
* **Discrete grid → implicit function**: with SIREN, the image *is* its parameters. Warping becomes an inverse-coordinate query and interpolation kernels disappear.

Every later geometry chapter — perspective projection (39), stereo (40), homographies (41), single-view metrology (42), structure-from-motion (44), NeRFs (45) — reuses the homogeneous algebra and the point/line duality set up here.